# LAB 08 - TravelOps
## Notebook: 00_seed_raw_data

Purpose:
Seeds selected `samples.wanderbricks` source tables into the Terraform-owned raw landing Volume.

Business purpose:
Creates repeatable raw travel booking data so CI/CD can prove the same application promotes from DEV to PROD.

Technical purpose:
Verifies the DAB target schema and Terraform-owned raw Volume, then writes each source table's deterministic sample **once** as an immutable Parquet file for Auto Loader, and explicitly fails if that content ever appears to have changed underneath an identity this notebook has declared immutable.

## Architecture decision: immutable, write-once seed (not versioned snapshots, not fingerprint-based change detection)

`samples.wanderbricks` is a fixed, Databricks-provided sample dataset, not a live business feed: this notebook's own stated purpose is to create *repeatable* raw data so CI/CD can prove promotion, and the table row counts documented in `README.md` (`bookings` 72,247, `payments` 49,638, etc.) have been stable across this project's entire history. All seven tables are sourced from this one static catalog via the same deterministic query pattern (order-by-key-and-limit, or booking-scoped filter) -- there is no genuine per-table difference in *ingestion* semantics, even though `payments`/`reviews` legitimately contain multiple business events sharing a foreign key, and `booking_updates` is business-modeled as an event layer over `bookings` in Silver. Those are correctly handled downstream (composite-key deduplication in `pipeline/silver.py`, the latest-row-per-`booking_id` window function in `current_bookings_silver`) regardless of how Bronze is seeded, because they describe multiplicity *within* one static snapshot, not a need to re-sample repeatedly.

Given that, this notebook treats each table's raw seed as **case A: a fixed, immutable dataset used to demonstrate deployment** -- not case B (versioned snapshots that can legitimately change across runs) or case C (genuine append-only events). A prior version of this notebook used a 64-bit `bit_xor(xxhash64(...))` fingerprint to decide whether content had changed. That was wrong on its own terms: XOR cancels in pairs, so a dataset with one copy of a row and a dataset with three copies of that same row can produce an *identical* fingerprint, silently failing to detect a multiplicity change -- and it was solving a problem (versioned snapshots) that this project does not actually have. Replacing it with a different hash would not fix the underlying issue: any change-detection scheme built for versioned snapshots still has to answer "what happens to a changed or deleted record," and composite-key deduplication in Silver does not answer that -- it only removes exact duplicate rows, it cannot un-write a stale row whose value changed or whose source row disappeared.

The design below instead makes each table's identity **explicit and human-controlled** (`SEED_VERSION` and `seed_limit`, both requiring a reviewed code/config change to alter -- never inferred from data content), and verifies that identity with an honest, multiplicity-aware but explicitly-limited signal: the row count captured in the seed's own file name.

Behavior by scenario:
- **First execution:** no file matches this identity's prefix yet; write it once (via a stage-then-move sequence that never touches an existing file).
- **Second execution, unchanged input:** a file already exists whose name encodes the current row count exactly; skip -- zero filesystem operations, genuinely idempotent.
- **Changed record / deleted record:** either changes the row count (caught: a file exists for this identity's prefix but not this row count -- the notebook raises and refuses to write, rather than silently keeping stale data or silently adding an incompatible second snapshot) or leaves the row count unchanged (a same-count in-place value edit -- **not caught**; explicitly documented below, not claimed as covered).
- **Legitimate duplicate/multiple events** (e.g. several payments for one booking): fully preserved -- the whole queried DataFrame is written verbatim in the one seed file; `pipeline/silver.py`'s composite-key deduplication is what correctly keeps these while collapsing only exact re-ingestion duplicates, and that logic is unaffected by this Bronze-layer redesign.
- **Interrupted write:** the new file only becomes visible under `raw/<table>/` via `dbutils.fs.mv` after being fully written and verified in a staging area outside the tree Auto Loader watches. A crash between staging and the move leaves no file at the final path, so a restarted run sees "not yet seeded" and retries cleanly -- this is what makes the design idempotent after a restart, not merely idempotent on paper.
- **Existing Auto Loader checkpoints:** never touched by this notebook. Once a file exists for an identity, no further file is ever written for that identity under normal operation, so Auto Loader's checkpoint stays completely undisturbed. A deliberate migration (bumping `SEED_VERSION` because the sampling *logic* changed, or accepting new `samples.wanderbricks` content) is a human, reviewed action that must separately decide whether to physically remove the old raw file and run the scoped Bronze full refresh already documented in `evidence/lab08_production_remediation_plan.md` -- this notebook does not attempt that automatically.
- **Historical Bronze duplicates from before this fix:** not removed by this change. The very first run of this corrected notebook against an already-affected schema will still add one more seed generation per table (no file yet matches the new `SEED_VERSION`-based naming), consistent with the duplication already documented in `evidence/lab08_photon_fix_and_reconciliation_observation.md`; from that point forward, reruns with unchanged content are true no-ops. Cleaning up what already exists remains the explicit, human-approved full-refresh procedure, not something this notebook does.
- **Checking that a file exists is not proof of ingestion:** this notebook only controls what is written to the raw Volume. Whether Auto Loader has actually picked up a given file, and whether `cloudFiles.allowOverwrites=false` behaves as documented, are Bronze/Lakeflow-side facts this notebook cannot observe or guarantee -- only a real Databricks run can confirm them (see `evidence/lab08_production_remediation_plan.md`'s validation procedure).

**Explicit, accepted limitation:** row count is multiplicity-aware (one copy vs. three copies of a row always differ in count, so the exact XOR failure mode above cannot recur), but it is not a full content check. A same-row-count change -- a value edited in place, or one row swapped for a different row while the total count stays the same -- is not detected, and this design does not claim otherwise. This is accepted specifically because the source is a fixed sample dataset where that scenario is not expected; if `samples.wanderbricks` were a genuinely live, mutable source, this design would need a stronger per-row content check (e.g. a canonical-sorted-and-chained hash, not XOR) or a different architecture entirely (case B or D).

Inputs:
- samples.wanderbricks.bookings
- samples.wanderbricks.booking_updates
- samples.wanderbricks.payments
- samples.wanderbricks.users
- samples.wanderbricks.properties
- samples.wanderbricks.reviews
- samples.wanderbricks.destinations

Outputs:
- /Volumes/<raw_volume_catalog>/<raw_volume_schema>/<raw_volume>/raw/<source_table>/

Tables/files affected:
Only raw Parquet folders under `raw/<source_table>/` are written to, and only once per `(SEED_VERSION, seed_limit)` identity: this notebook never deletes an existing file there, and never writes a second file for an identity it has already seeded successfully. Schema and Volume metadata are owned by Terraform.

Environment variables/widgets used:
`target_catalog`, `target_schema`, `raw_volume_catalog`, `raw_volume_schema`, `raw_volume_name`, `raw_volume_type`, `raw_volume_storage_location`, `source_catalog`, `source_schema`, `seed_limit`. `SEED_VERSION` is a code constant, not a widget, so changing it requires a reviewed code change.

Creates/modifies data:
Yes, but only once per identity. A rerun with the same `SEED_VERSION`/`seed_limit` and unchanged upstream row count writes nothing. A rerun whose row count differs under the same identity raises an exception instead of writing anything.

Dependencies/prerequisites:
The DAB target schema must exist, Terraform must have already created the raw Volume, and the deployer must have read access to `samples.wanderbricks`.

Expected result:
Each configured source table has exactly one raw Parquet file per `(SEED_VERSION, seed_limit)` identity, named with its row count, in the target Volume. `bookings`, `users`, `properties` and `destinations` are sampled independently by row order. `booking_updates`, `payments` and `reviews` are booking-scoped event tables, so they are instead filtered to the exact `booking_id` set sampled from `bookings`, keeping every related event for a sampled booking instead of an independent, referentially-inconsistent row-count cap.

Failure behavior:
The notebook fails fast when required parameters are missing, when the Terraform-owned raw Volume is absent, when source/target privileges are insufficient, or when a table's sampled row count no longer matches the count recorded for its current `SEED_VERSION`/`seed_limit` identity (see the architecture decision above).

Environment classification:
Safe for personal_dev and personal_prod runs. Azure PROD job execution is intentionally deferred until explicitly authorized.


### Step 1 - Resolve target configuration

This cell reads Databricks widgets supplied by the Bundle job. It validates required values before any write occurs so configuration errors fail before partial data is created.

In [ ]:
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")
dbutils.widgets.text("raw_volume_catalog", "")
dbutils.widgets.text("raw_volume_schema", "")
dbutils.widgets.text("raw_volume_name", "lab08_dev_travelops_raw")
dbutils.widgets.text("raw_volume_type", "MANAGED")
dbutils.widgets.text("raw_volume_storage_location", "")
dbutils.widgets.text("source_catalog", "samples")
dbutils.widgets.text("source_schema", "wanderbricks")
dbutils.widgets.text("seed_limit", "25000")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
raw_volume_catalog = dbutils.widgets.get("raw_volume_catalog")
raw_volume_schema = dbutils.widgets.get("raw_volume_schema")
raw_volume_name = dbutils.widgets.get("raw_volume_name")
raw_volume_type = dbutils.widgets.get("raw_volume_type").upper()
raw_volume_storage_location = dbutils.widgets.get("raw_volume_storage_location")
source_catalog = dbutils.widgets.get("source_catalog")
source_schema = dbutils.widgets.get("source_schema")
seed_limit = int(dbutils.widgets.get("seed_limit"))

required = {
    "target_catalog": target_catalog,
    "target_schema": target_schema,
    "raw_volume_catalog": raw_volume_catalog,
    "raw_volume_schema": raw_volume_schema,
    "raw_volume_name": raw_volume_name,
    "source_catalog": source_catalog,
    "source_schema": source_schema,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError(f"Missing required widgets: {missing}")
if raw_volume_type == "EXTERNAL" and not raw_volume_storage_location:
    raise ValueError("External raw volume requires raw_volume_storage_location")


### Step 2 - Verify target schema and Terraform-owned raw Volume

This cell performs read-only metadata checks. It intentionally does not create or alter the raw Volume schema or raw Volume because Terraform owns that layer; the application target schema is checked separately.

In [ ]:
target_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{target_catalog}` LIKE '{target_schema}'").count() == 1
if not target_schema_exists:
    raise ValueError(f"Required DAB target schema does not exist: {target_catalog}.{target_schema}")

raw_schema_exists = spark.sql(f"SHOW SCHEMAS IN `{raw_volume_catalog}` LIKE '{raw_volume_schema}'").count() == 1
if not raw_schema_exists:
    raise ValueError(f"Required Terraform-referenced raw schema does not exist: {raw_volume_catalog}.{raw_volume_schema}")

volume_rows = spark.sql(f"SHOW VOLUMES IN `{raw_volume_catalog}`.`{raw_volume_schema}` LIKE '{raw_volume_name}'").collect()
if len(volume_rows) != 1:
    raise ValueError(f"Required Terraform-owned raw Volume is missing: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")

print(f"Verified DAB target schema: {target_catalog}.{target_schema}")
print(f"Verified Terraform-owned raw Volume: {raw_volume_catalog}.{raw_volume_schema}.{raw_volume_name}")


### Step 3 - Seed deterministic raw Parquet folders

This cell reads the discovered Wanderbricks tables and overwrites deterministic target folders. Overwrite mode prevents uncontrolled duplicate files across reruns while keeping the raw contract file-based for Auto Loader.

In [ ]:
SEED_VERSION = "v1"
# A small, explicit, human-controlled version marker. Bump this only as a
# deliberate part of a reviewed migration when the sampling LOGIC itself
# changes (e.g. a different BOOKING_SCOPED_TABLES membership or ordering
# column) -- never inferred automatically from data content. See the
# architecture decision in this notebook's introduction cell.

source_tables = [
    "bookings",
    "booking_updates",
    "payments",
    "users",
    "properties",
    "reviews",
    "destinations",
]

# booking_updates, payments and reviews reference booking_id. Sampling each
# of these independently by row order (as bookings/users/properties/
# destinations still are) breaks that relationship: a booking's related
# payment or update rows can fall outside a same-sized but differently
# ordered independent sample, which previously produced "missing payment"
# rows caused purely by sampling rather than by the source data. Sampling
# bookings first and then filtering these three tables to that exact
# booking_id set keeps every related event for a sampled booking.
BOOKING_SCOPED_TABLES = {"booking_updates", "payments", "reviews"}

bookings_source = f"`{source_catalog}`.`{source_schema}`.`bookings`"
bookings_df = spark.table(bookings_source)
bookings_sample_df = bookings_df.orderBy(*bookings_df.columns[:1]).limit(seed_limit)
sampled_booking_ids_df = bookings_sample_df.select("booking_id")


def _seed_identity_prefix(current_seed_limit: int) -> str:
    """Human-controlled identity prefix for a table's seed: version + seed_limit.

    Deliberately not derived from data content. SEED_VERSION and seed_limit
    are the only two things allowed to define a new seed identity, and both
    require a deliberate, reviewed code/config change to alter.
    """

    return f"seed-{SEED_VERSION}-{current_seed_limit}-"


def _seed_file_name(current_seed_limit: int, row_count: int) -> str:
    """Deterministic file name for a table's seed at a given identity and row count.

    Encoding row_count directly in the name is the verification signal:
    unlike a hash-based fingerprint, it cannot cancel out a multiplicity
    change (one copy vs. three copies of a row always produce different
    counts), but it is an explicitly limited signal -- a same-row-count
    in-place value edit is not detected. See this notebook's architecture
    decision for why that trade-off is accepted here.
    """

    return f"{_seed_identity_prefix(current_seed_limit)}{row_count}rows.snappy.parquet"


def _existing_seed_files(target_path: str, prefix: str):
    """List existing seed data files under target_path matching this identity's prefix.

    Never deletes or modifies anything. A missing target_path (first-ever
    seed for this table) is treated as "nothing exists yet," not an error --
    but only that specific, expected condition. An authentication failure,
    permission error or transient storage failure must not be silently
    treated the same way: doing so would make this function report "no file
    exists" for a table whose seed file is actually present but temporarily
    unlistable, which would then cause _write_immutable_seed to run again
    and dbutils.fs.mv to silently overwrite that already-ingested immutable
    file. Only the known "path does not exist yet" error is swallowed;
    everything else propagates and stops the notebook.
    """

    try:
        listing = dbutils.fs.ls(target_path)
    except Exception as e:
        if "java.io.FileNotFoundException" in str(e) or "FileNotFoundException" in str(e):
            return []
        raise

    return [f for f in listing if f.name.startswith(prefix) and f.name.endswith(".snappy.parquet")]


def _write_immutable_seed(df, target_path: str, staging_root: str, table_name: str, file_name: str) -> None:
    """Write df once as a new immutable Parquet file under target_path.

    Only ever called when no file exists yet for this exact identity and
    row count. Stages the write outside the raw/<table>/ tree Auto Loader
    watches, verifies a single part file was produced, then moves it into
    place -- so Auto Loader never observes a partial write, and a crash
    between staging and the move leaves no file at the final path, which a
    restarted run correctly treats as "not yet seeded" and retries cleanly.
    Never deletes an existing file.

    Concurrent executions of this notebook for the same identity are
    prevented at the job level (resources/job.yml sets
    max_concurrent_runs: 1 on travelops_promotion_job), so this function
    does not implement its own locking against that race.
    """

    staging_path = f"{staging_root}/{table_name}"
    try:
        dbutils.fs.rm(staging_path, recurse=True)
    except Exception:
        pass

    df.coalesce(1).write.mode("overwrite").format("parquet").save(staging_path)
    part_files = [f.path for f in dbutils.fs.ls(staging_path) if f.name.startswith("part-")]
    if len(part_files) != 1:
        raise AssertionError(
            f"Expected exactly one staged part file for {table_name}, found {len(part_files)}: {part_files}"
        )

    final_path = f"{target_path}/{file_name}"
    dbutils.fs.mv(part_files[0], final_path)
    dbutils.fs.rm(staging_path, recurse=True)


staging_root = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/_seed_staging"

seed_summary = []
for table_name in source_tables:
    source_table = f"`{source_catalog}`.`{source_schema}`.`{table_name}`"
    target_path = f"/Volumes/{raw_volume_catalog}/{raw_volume_schema}/{raw_volume_name}/raw/{table_name}"
    if table_name == "bookings":
        df = bookings_sample_df
    elif table_name in BOOKING_SCOPED_TABLES:
        df = spark.table(source_table).join(sampled_booking_ids_df, "booking_id", "left_semi")
    else:
        source_df = spark.table(source_table)
        df = source_df.orderBy(*source_df.columns[:1]).limit(seed_limit)

    row_count = df.count()
    prefix = _seed_identity_prefix(seed_limit)
    file_name = _seed_file_name(seed_limit, row_count)
    existing_files = _existing_seed_files(target_path, prefix)
    final_path = f"{target_path}/{file_name}"

    if not existing_files:
        _write_immutable_seed(df, target_path, staging_root, table_name, file_name)
        seed_status = "written (first seed for this SEED_VERSION/seed_limit)"
    elif any(f.name == file_name for f in existing_files):
        seed_status = "unchanged (write skipped)"
    else:
        # Same SEED_VERSION/seed_limit identity, but a different row count
        # than when this identity was first seeded: samples.wanderbricks
        # content changed underneath an identity declared immutable. Fail
        # loudly instead of silently keeping the stale file or silently
        # writing a second, additive snapshot with no supersession
        # semantics.
        raise AssertionError(
            f"Unexpected content change detected for {table_name}: existing seed file(s) "
            f"{[f.name for f in existing_files]} do not match the current row count "
            f"({row_count}) for SEED_VERSION={SEED_VERSION}, seed_limit={seed_limit}. "
            "samples.wanderbricks appears to have changed under an identity that was "
            "declared immutable. Resolve this with a deliberate, reviewed migration "
            "(bump SEED_VERSION and follow the Bronze full-refresh procedure in "
            "evidence/lab08_production_remediation_plan.md), not by rerunning this notebook."
        )

    seed_summary.append((table_name, row_count, final_path, seed_status))

display(
    spark.createDataFrame(
        seed_summary, "table_name STRING, row_count LONG, target_path STRING, seed_status STRING"
    )
)
